In [58]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf
import numpy as np
import pandas as pd
from keras.models import Model, Sequential
from keras.layers import Input, Dense, Conv1D, Flatten, MaxPooling1D, Concatenate, Dropout
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split

In [59]:
%matplotlib inline

In [60]:
seed_value = 42
os.environ['PYTHONHASHSEED'] = str(seed_value)
np.random.seed(seed_value)
tf.random.set_seed(seed_value)

In [61]:
value_df = 10000
star_check = pd.read_csv("./Datasets/updated_database_exoplanet.csv")
star_check = star_check.drop(['tid'],axis=1)
star_check_y = star_check[['confirmed_planet']]
star_check = star_check.reset_index().drop('index',axis=1)
star_check = star_check.apply(lambda row: row.fillna(0), axis=1)

In [62]:
scaler = MinMaxScaler()
star_check[['Teff','logg','MH','rad','mass','rho','lum','Tmag','ra','dec','plx']] = scaler.fit_transform(star_check[['Teff','logg','MH','rad','mass','rho','lum','Tmag','ra','dec','plx']])

In [63]:
X_train, X_test, y_train, y_test = train_test_split(star_check.drop('confirmed_planet',axis=1),star_check[['confirmed_planet']], test_size=0.1, random_state=42)

In [64]:
X_train_flux = X_train.drop(['Teff','logg','MH','rad','mass','rho','lum','Tmag','ra','dec','plx'],axis=1)
X_train_params = X_train[['Teff','logg','MH','rad','mass','rho','lum','Tmag','ra','dec','plx']]
X_test_flux = X_test.drop(['Teff','logg','MH','rad','mass','rho','lum','Tmag','ra','dec','plx'],axis=1)
X_test_params = X_test[['Teff','logg','MH','rad','mass','rho','lum','Tmag','ra','dec','plx']]

In [65]:
input_flux = Input(shape=(10000,1))
x = Conv1D(filters=32, kernel_size=5, activation='relu')(input_flux)
x = MaxPooling1D(pool_size=2)(x)
x = Flatten()(x)
model_flux = Model(inputs=input_flux, outputs=x)

input_params = Input(shape=(11,))
y = Dense(128, activation='relu')(input_params)
model_params = Model(inputs=input_params, outputs=y)

combined = Concatenate()([model_flux.output, model_params.output])
z = Dense(128, activation='relu')(combined)
final_output = Dense(1, activation='sigmoid')(z)

model = Model(inputs=[model_flux.input, model_params.input], outputs=final_output)

In [66]:
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',  
    patience=10,        
    mode='min'          
)

class MyThresholdCallback(tf.keras.callbacks.Callback):
    def __init__(self, threshold):
        super(MyThresholdCallback, self).__init__()
        self.threshold = threshold

    def on_epoch_end(self, epoch, logs=None):
        val_acc = logs["val_accuracy"]
        acc = logs['accuracy']
        if val_acc >= self.threshold and acc >= self.threshold:
            self.model.stop_training = True

my_callback = MyThresholdCallback(threshold=0.9)

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# history = model.fit([X_train_flux, X_train_params], y_train, epochs=100, batch_size=32, validation_split=0.2, callbacks=[my_callback, early_stopping])

In [67]:
model.load_weights('./Models/model_weights.h5')

In [68]:
loss, accuracy = model.evaluate([X_test_flux,X_test_params],y_test)
print(f"Accuracy: {accuracy*100}%")

22/22 ━━━━━━━━━━━━━━━━━━━━ 1s 26ms/step - accuracy: 0.8894 - loss: 0.3394
Accuracy: 90.04328846931458%
